In [2]:
from transformers import BigBirdTokenizer
import pandas as pd
import os
from tqdm import tqdm
import json

/home/mario/miniforge3/envs/xtemp-nlp/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
model_id = 'google/bigbird-roberta-base'
tokenizer = BigBirdTokenizer.from_pretrained(model_id, trust_remote_code=True)

## Domain Adaptation Data - Baseline

In [ ]:
def convert_to_csv(input_path, output_path, tokenizer):
    assert os.path.exists(input_path)

    if not os.path.exists(output_path):
        os.makedirs(output_path)

    all_tokens_da = 0
    for train_or_dev in tqdm(os.listdir(input_path), 'Conversion to csv...'):
        train_or_dev_path = os.path.join(input_path, train_or_dev)

        df = {'entries': []}

        with open(train_or_dev_path, 'r') as f:
            objects = f.read().strip().split('\n')
            entries = [json.loads(obj) for obj in objects]

            raw_corpus = ""
            for entry in entries:
                if entry['exp_to_edit'] is None:
                    continue
                else:
                    if entry['exp_to_edit']:
                        if 'exp_upd' in entry and entry['exp_upd'] is not None:
                            raw_corpus += entry['exp_upd'] + '\n\n'
                            df['entries'].append(entry['exp_upd'])
                    else:
                        raw_corpus += entry['exp'] + '\n\n'
                        df['entries'].append(entry['exp'])

            raw_corpus = raw_corpus.strip()
            tokens_corpus = tokenizer.tokenize(raw_corpus)

            print(
                f'\n\n{train_or_dev} has {round(len(tokens_corpus) / (10 ** 6), 4)}M tokens for domain adaptation\n\n')

            all_tokens_da += len(tokens_corpus)

        df = pd.DataFrame(df)

        savename = 'dev.csv' if 'dev' in train_or_dev else 'train.csv'

        df.to_csv(os.path.join(output_path, savename), index=False)

    print(f'Total of {round(all_tokens_da / (10 ** 9), 4)}B tokens for domain adaptation')


convert_to_csv('raw_baseline', '../data/baseline', tokenizer)

## QA Dataset

In [ ]:
with open('raw_baseline/dev.jsonl', 'r') as f:
    lines = f.read().strip().split('\n')
    dev = [json.loads(line) for line in lines]

    test_data = []
    for test_example in tqdm(dev, desc='converting to eval QA dataset'):
        test_dict = {
            'question': test_example['question_upd'],
            'subject_name': test_example['subject_name'],
            'cop': test_example['cop'],
            'opa': test_example['opa_upd'],
            'opb': test_example['opb_upd'],
            'opc': test_example['opc_upd'],
            'opd': test_example['opd_upd']
        }
        test_data.append(test_dict)

with open('baseline/test.json', 'w') as fw:
    json.dump(test_data, fw, indent=2)

## Domain Adaptation Data - Conflicts

In [ ]:
def convert_to_csv(input_path, output_path, tokenizer):
    assert os.path.exists(input_path)

    if not os.path.exists(output_path):
        os.makedirs(output_path)

    all_tokens_da = 0
    for train_or_dev in tqdm(os.listdir(input_path), 'Conversion to csv...'):
        if train_or_dev != 'train_conflict.json':
            continue
        train_or_dev_path = os.path.join(input_path, train_or_dev)

        df = {'entries': []}

        with open('raw_baseline/train.jsonl', 'r') as f:
            lines = f.read().strip().split('\n')
            clean_data = [json.loads(l) for l in lines]

        unique_questions = set()
        with open(train_or_dev_path, 'r') as f:
            entries = json.load(f)

            ## take all conflicts
            raw_corpus = ""
            for entry in entries:
                unique_questions.add(entry['question'])
                raw_corpus += entry['mod_context'] + '\n\n'

                df['entries'].append(entry['mod_context'])

            ## sample the cleaned contexts from the baseline data
            ## which do not appear in the conflicting data
            for entry in clean_data:
                if entry['question'] not in unique_questions:
                    if entry['exp_to_edit'] is None:
                        continue
                    else:
                        if entry['exp_to_edit']:
                            if 'exp_upd' in entry and entry['exp_upd'] is not None:
                                raw_corpus += entry['exp_upd'] + '\n\n'
                                df['entries'].append(entry['exp_upd'])
                        else:
                            raw_corpus += entry['exp'] + '\n\n'
                            df['entries'].append(entry['exp'])

            raw_corpus = raw_corpus.strip()
            tokens_corpus = tokenizer.tokenize(raw_corpus)

            print(
                f'\n\n{train_or_dev} has {round(len(tokens_corpus) / (10 ** 6), 4)}M tokens for domain adaptation\n\n')

            all_tokens_da += len(tokens_corpus)

        df = pd.DataFrame(df)

        savename = 'dev.csv' if 'dev' in train_or_dev else 'train.csv'

        df.to_csv(os.path.join(output_path, savename), index=False)

    print(f'Total of {round(all_tokens_da / (10 ** 9), 4)}B tokens for domain adaptation')


convert_to_csv('raw_conflicting', '../data/conflicting', tokenizer)